##### Creating a parameter map

This notebook is meant to show the user how to create a parameter map to see what the spatial distribution of learned parameters looks like

In [ ]:
# Run imports
import logging
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import yaml
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from mpl_toolkits.axes_grid1 import make_axes_locatable

from ddr._version import __version__
from ddr.geodatazoo.lynker_hydrofabric import LynkerHydrofabric
from ddr.io.readers import ForcingsReader
from ddr.io.readers import StreamflowReader as streamflow
from ddr.nn import CudaLSTM, kan
from ddr.routing.torch_mc import dmc
from ddr.routing.utils import denormalize
from ddr.scripts_utils import load_checkpoint
from ddr.validation import Config

log = logging.getLogger(__name__)

In [ ]:
# Load a config
with open("./example_config.yaml") as f:
    config = Config(**yaml.safe_load(f))

In [ ]:
# Instantiate objects required for routing
device = torch.device(f"cuda:{config.device}")
nn = kan(
    input_var_names=config.kan.input_var_names,
    learnable_parameters=config.kan.learnable_parameters,
    hidden_size=config.kan.hidden_size,
    num_hidden_layers=config.kan.num_hidden_layers,
    grid=config.kan.grid,
    k=config.kan.k,
    seed=config.seed,
    device=config.device,
    gate_parameters=config.kan.gate_parameters,
    off_parameters=config.kan.off_parameters,
    use_graph_context=config.kan.use_graph_context,
    output_embedding=config.kan.use_node_processor,
)
routing_model = dmc(cfg=config, device=device)
flow = streamflow(config)
dataset = config.geodataset.get_dataset_class(cfg=config)

# Instantiate LSTM + ForcingsReader only when configured (produces d_gw when leakance enabled)
lstm_nn = None
forcings_reader = None
if config.cuda_lstm is not None:
    lstm_nn = CudaLSTM(
        input_var_names=config.cuda_lstm.input_var_names,
        forcing_var_names=config.cuda_lstm.forcing_var_names,
        learnable_parameters=config.cuda_lstm.learnable_parameters,
        hidden_size=config.cuda_lstm.hidden_size,
        num_layers=config.cuda_lstm.num_layers,
        dropout=config.cuda_lstm.dropout,
        seed=config.seed,
        device=config.device,
    )
    forcings_reader = ForcingsReader(config)

In [ ]:
# Load pretrained model states (KAN + routing GNN submodules + LSTM)
model_states = Path(config.experiment.checkpoint)
load_checkpoint(
    nn,
    model_states,
    device,
    lstm_nn=lstm_nn,
    routing_model=routing_model if config.kan.use_node_processor else None,
)

In [ ]:
# Generate KAN parameters
# In GNN mode (use_node_processor=True), KAN returns an embedding tensor h^0.
# We decode it via ParamDecoder to get the initial (t=0) parameter snapshot.
# In classic mode, KAN returns a dict of [0,1] sigmoid outputs directly.
nn = nn.eval()
routing_model.eval()
with torch.no_grad():
    kan_out = nn(inputs=dataset.routing_dataclass.normalized_spatial_attributes.to(device))

    if config.kan.use_node_processor:
        # GNN mode: decode embedding → parameter dict via ParamDecoder
        spatial_params = routing_model.param_decoder(kan_out)
    else:
        spatial_params = kan_out

    # Denormalize to physical values
    for key in spatial_params:
        if key in config.params.parameter_ranges:
            log_space = key in config.params.log_space_parameters
            spatial_params[key] = denormalize(
                spatial_params[key],
                config.params.parameter_ranges[key],
                log_space=log_space,
            )

# Move to CPU float16 to free VRAM for LSTM batching
spatial_params = {k: v.cpu().half() for k, v in spatial_params.items()}
torch.cuda.empty_cache()

In [ ]:
# Generate LSTM parameters (d_gw when leakance enabled) — batched to avoid GPU OOM
BATCH_SIZE = 1_000

if lstm_nn is not None and forcings_reader is not None and config.params.use_leakance:
    lstm_nn = lstm_nn.eval()

    # Load forcings to CPU (~790 MB for full CONUS, avoids GPU OOM)
    with torch.no_grad():
        all_forcings = forcings_reader(
            routing_dataclass=dataset.routing_dataclass, device="cpu", dtype=torch.float32
        )
    all_attributes = dataset.routing_dataclass.normalized_spatial_attributes  # (N, num_attrs), CPU

    N = all_attributes.shape[0]
    d_gw_cpu = torch.zeros(N, dtype=torch.float16)

    with torch.no_grad():
        for start in range(0, N, BATCH_SIZE):
            end = min(start + BATCH_SIZE, N)
            batch_forcings = all_forcings[:, start:end, :].to(device)
            batch_attrs = all_attributes[start:end, :].to(device)

            batch_out = lstm_nn(forcings=batch_forcings, attributes=batch_attrs)

            if "d_gw" in batch_out:
                d_gw_cpu[start:end] = (
                    denormalize(batch_out["d_gw"].mean(dim=0), config.params.parameter_ranges["d_gw"])
                    .cpu()
                    .half()
                )

            del batch_forcings, batch_attrs, batch_out
            torch.cuda.empty_cache()

    del all_forcings
    spatial_params["d_gw"] = d_gw_cpu

In [ ]:
# Map parameters back to the hydrofabric
if config.geodataset == "lynker_hydrofabric":
    gdf = gpd.read_file(config.data_sources.geospatial_fabric_gpkg, layer="divides").set_index("divide_id")
    divide_ids = np.array([f"cat-{_id}" for _id in dataset.hf_ids])
    gdf = gdf.reindex(divide_ids)
elif config.geodataset == "merit":
    gdf = gpd.read_file(config.data_sources.geospatial_fabric_gpkg).set_index("COMID")
    divide_ids = np.array(dataset.routing_dataclass.divide_ids)
    gdf = gdf.loc[divide_ids]
else:
    raise ValueError(f"Unsupported geodataset: {config.geodataset}")

# Add all decoded parameters to the GeoDataFrame
for key, values in spatial_params.items():
    gdf[key] = values.float().numpy()

gdf = gdf.to_crs(epsg=4326)
print(f"Parameters available for plotting: {list(spatial_params.keys())}")

In [ ]:
def param_plot(
    gdf: gpd.GeoDataFrame,
    var: str,
    save_name: Path,
    cmap: str = "plasma",
    unit_label: str | None = None,
    title: str | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    ascending: bool = False,
    dpi: int = 100,
) -> tuple[Figure, Axes]:
    """
    Create a parameter plot for geospatial data with a basemap and colorbar.

    Parameters
    ----------
    gdf : gpd.GeoDataFrame
        GeoDataFrame containing the data to plot.
    var : str
        Column name to visualize.
    save_name : str
        Filename for saving the plot.
    cmap : str, default 'plasma'
        Colormap name for the plot.
    unit_label : str, optional
        Unit label for the colorbar.
    title : str, optional
        Title for the plot.
    vmin : float, optional
        Minimum value for color scaling. If None, uses data minimum.
    vmax : float, optional
        Maximum value for color scaling. If None, uses data maximum.
    ascending : bool, default False
        Whether to sort data in ascending order.
    dpi : int, default 100
        DPI for the figure display.

    Returns
    -------
    tuple of (matplotlib.figure.Figure, matplotlib.axes.Axes)
        Figure and axes objects for further customization.

    Raises
    ------
    KeyError
        If the specified variable column doesn't exist in the GeoDataFrame.
    ValueError
        If the GeoDataFrame is empty after dropping NaN values.

    """
    # Validate inputs
    if var not in gdf.columns:
        raise KeyError(f"Column '{var}' not found in GeoDataFrame")

    # Create figure
    fig, ax = plt.subplots(figsize=(7, 4), dpi=dpi)

    # Drop NaNs and validate data
    gdf_clean = gdf.dropna(subset=[var])
    if gdf_clean.empty:
        raise ValueError(f"No valid data found for variable '{var}' after dropping NaN values")

    # Sort data for visualization
    gdf_clean = gdf_clean.sort_values(by=var, ascending=ascending)
    data = gdf_clean[var].values

    # Set vmin and vmax if not provided
    if vmin is None:
        vmin = np.min(data)
    if vmax is None:
        vmax = np.nanmax(data)

    # Create the plot with direct vmin/vmax limits
    gdf_clean.plot(
        ax=ax,
        column=var,
        cmap=cmap,
        linewidth=0.3,
        vmin=vmin,
        vmax=vmax,
        zorder=1,
    )

    # Add basemap
    cx.add_basemap(
        ax,
        crs=gdf_clean.crs,
        source=cx.providers.CartoDB.Positron,
        alpha=0.6,
        zorder=0,
        attribution=False,
    )

    # Set bounds for CONUS
    ax.set_xlim(-125, -66)
    ax.set_ylim(24, 53)

    # Remove axis ticks
    ax.set_xticks([])
    ax.set_yticks([])

    # Set plot title
    if title is not None:
        ax.set_title(title, fontsize=14)

    # Add colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.1)
    sm = plt.cm.ScalarMappable(cmap=cmap)
    sm.set_array([])
    sm.set_clim(vmin, vmax)
    cbar = fig.colorbar(sm, cax=cax)

    # Set colorbar label
    label_text = var
    if unit_label:
        label_text = f"{var} ({unit_label})"
    cbar.set_label(label_text)

    # Format tick values to show appropriate precision
    cbar.formatter.set_powerlimits((-2, 2))
    cbar.update_ticks()

    # Save figure
    plt.tight_layout()
    # Note: config.params.save_path reference removed - you'll need to pass the full path
    # or import your config module
    plt.savefig(save_name, dpi=600, bbox_inches="tight")

    return fig, ax

In [ ]:
# Plot parameters
param_plot(
    gdf,
    "n",
    Path(config.params.save_path) / "n_train.png",
    vmax=0.2,
    cmap="plasma_r",
    title="Manning's Roughness (m⁻¹/³s)",
    ascending=True,
    dpi=200,
)

In [ ]:
# param_plot(
#     gdf,
#     "side_slope",
#     Path(config.params.save_path) / "side_slope_train.png",
#     cmap="plasma_r",
#     title="side_slope",
#     ascending=True,
#     dpi=200,
# )

In [ ]:
# Plot K_D — hydraulic exchange rate (leakance, only present when use_leakance=True)
if "K_D" in gdf.columns:
    param_plot(
        gdf,
        "K_D",
        Path(config.params.save_path) / "K_D_train.png",
        cmap="PuRd",
        title="Hydraulic Exchange Rate (K_D)",
        unit_label="1/s",
        ascending=True,
        dpi=200,
    )

In [ ]:
# Plot d_gw — depth to water table (leakance, only present when use_leakance=True)
if "d_gw" in gdf.columns:
    param_plot(
        gdf,
        "d_gw",
        Path(config.params.save_path) / "d_gw_train.png",
        cmap="YlGnBu",
        title="Depth to Water Table (d_gw)",
        unit_label="m",
        ascending=True,
        dpi=200,
    )

In [ ]:
# Scatter plot: parameter vs upstream area (if available)
plot_var = "n" if "n" in spatial_params else list(spatial_params.keys())[0]
gdf_plot = gdf.loc[divide_ids]
gdf_plot[plot_var] = spatial_params[plot_var].float().numpy()
if "uparea" in gdf_plot.columns:
    plt.scatter(np.log(gdf_plot["uparea"]), gdf_plot[plot_var], alpha=0.3)
    plt.xlabel("log(upstream area)")
    plt.ylabel(plot_var)